In [1]:

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer

print("Libraries imported successfully!")
print("PyTorch version:", torch.__version__)


Libraries imported successfully!
PyTorch version: 2.11.0+cpu


In [2]:

# Create a small dataset of positive and negative reviews

texts = [
    "This movie was amazing",
    "I really loved this film",
    "The story was wonderful",
    "Excellent acting and great music",
    "I enjoyed this movie a lot",
    "The film was fantastic",
    "This is a great movie",
    "The movie was boring",
    "I hated this film",
    "The story was terrible",
    "Very bad acting",
    "I did not enjoy this movie",
    "The film was disappointing",
    "This movie was awful",
    "I disliked the story",
    "The movie was good",
    "The film was bad",
    "Amazing acting",
    "Terrible movie",
    "Wonderful story"
]

labels = [
    1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0,
    1, 0, 1, 0, 1
]

# Create a DataFrame
df = pd.DataFrame({
    "text": texts,
    "label": labels
})

print("Dataset created successfully!")
print("Total reviews:", len(df))
display(df.head())


Dataset created successfully!
Total reviews: 20


,text,label
0,This movie was amazing,1
1,I really loved this film,1
2,The story was wonderful,1
3,Excellent acting and great music,1
4,I enjoyed this movie a lot,1


In [3]:
# Separate text and labels
X = df["text"]
y = df["label"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 15
Testing samples: 5


In [4]:
# Convert text into numerical word-count features
vectorizer = CountVectorizer()

# Learn vocabulary only from training text
X_train_vectorized = vectorizer.fit_transform(X_train)

# Convert testing text using the same vocabulary
X_test_vectorized = vectorizer.transform(X_test)

# Convert features to PyTorch tensors
X_train_tensor = torch.tensor(
    X_train_vectorized.toarray(),
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_vectorized.toarray(),
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.long
)

print("Text converted into numerical features!")
print("Training tensor shape:", X_train_tensor.shape)
print("Testing tensor shape:", X_test_tensor.shape)

Text converted into numerical features!
Training tensor shape: torch.Size([15, 23])
Testing tensor shape: torch.Size([5, 23])


In [5]:
class TextClassifier(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.layer1 = nn.Linear(input_size, 16)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(16, 2)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x


# Create the model
input_size = X_train_tensor.shape[1]

model = TextClassifier(input_size)

print(model)

TextClassifier(
  (layer1): Linear(in_features=23, out_features=16, bias=True)
  (relu): ReLU()
  (output_layer): Linear(in_features=16, out_features=2, bias=True)
)


In [6]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

epochs = 50

# Train the neural network
for epoch in range(epochs):

    model.train()

    # Forward pass
    outputs = model(X_train_tensor)

    # Calculate loss
    loss = criterion(outputs, y_train_tensor)

    # Clear previous gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

print("Model training completed!")

Epoch [10/50], Loss: 0.5642
Epoch [20/50], Loss: 0.2996
Epoch [30/50], Loss: 0.0998
Epoch [40/50], Loss: 0.0275
Epoch [50/50], Loss: 0.0094
Model training completed!


In [7]:
# Evaluate on testing data
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)

    # Select the class with the highest score
    predictions = torch.argmax(test_outputs, dim=1)

y_true = y_test_tensor.numpy()
y_pred = predictions.numpy()

print("Classification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         3
    Positive       0.25      0.50      0.33         2

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5

Confusion Matrix:
[[0 3]
 [1 1]]


In [8]:
def predict_sentiment(review):

    # Convert the review using the trained vectorizer
    review_vector = vectorizer.transform([review])

    # Convert to a PyTorch tensor
    review_tensor = torch.tensor(
        review_vector.toarray(),
        dtype=torch.float32
    )

    # Predict sentiment
    model.eval()

    with torch.no_grad():
        output = model(review_tensor)
        predicted_class = torch.argmax(output, dim=1).item()

    sentiment = "Positive" if predicted_class == 1 else "Negative"

    print("Review:", review)
    print("Predicted sentiment:", sentiment)


# Try sample reviews
predict_sentiment("This movie was wonderful")
predict_sentiment("The film was terrible")
predict_sentiment("I really enjoyed this amazing movie")

Review: This movie was wonderful
Predicted sentiment: Positive
Review: The film was terrible
Predicted sentiment: Negative
Review: I really enjoyed this amazing movie
Predicted sentiment: Positive


In [9]:
# Save the trained neural network and vocabulary
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_size": input_size,
        "vocabulary": vectorizer.vocabulary_
    },
    "nlp_text_classifier.pth"
)

print("Model saved successfully!")

Model saved successfully!


In [10]:
from google.colab import files

files.download("nlp_text_classifier.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>